# Défi Quotidien : Visualisation interactive des données avec Matplotlib et Seaborn



## 1. Préparation et Prétraitement des Données
Nous chargeons les bibliothèques indispensables, puis nous mettons en place un jeu de données simulant fidèlement le comportement du *US Superstore* (Ventes, Profits, Remises, États, et Dates sur plusieurs années).

In [ ]:
# Activation de l'interactivité pour Matplotlib dans l'environnement Jupyter
%matplotlib notebook

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Génération d'un jeu de données Superstore réaliste
np.random.seed(42)
n_rows = 1200

etats_usa = ['California', 'New York', 'Texas', 'Washington', 'Florida', 'Illinois', 'Ohio', 'Pennsylvania']
produits = [f'Produit {chr(65+i)}{j}' for i in range(5) for j in range(1, 5)] # Ex: Produit A1, B3...

data = {
    'ID Ligne': range(1, n_rows + 1),
    'Date Commande': pd.date_range(start='2023-01-01', periods=n_rows, freq='12H'),
    'État': np.random.choice(etats_usa, n_rows, p=[0.25, 0.20, 0.15, 0.10, 0.10, 0.08, 0.06, 0.06]),
    'Produit': np.random.choice(produits, n_rows),
    'Ventes': np.round(np.random.exponential(scale=250, size=n_rows) + 10, 2),
    'Remise': np.random.choice([0.0, 0.1, 0.2, 0.4, 0.6], n_rows, p=[0.4, 0.2, 0.2, 0.1, 0.1])
}

df = pd.DataFrame(data)

# Calcul du profit en fonction des ventes et des remises
df['Profit'] = np.round(df['Ventes'] * (0.35 - df['Remise']) + np.random.normal(0, 15, n_rows), 2)

# Prétraitement et extraction des variables temporelles
df['Année-Mois'] = df['Date Commande'].dt.to_period('M')

print("Données Superstore prêtes ! Aperçu du fichier nettoyé :")
df.head()

## 2. Visualisation des Données avec Matplotlib (Mode Interactif)

### A. Graphique linéaire interactif : Tendances des ventes au fil des mois

In [ ]:
# Agrégation des ventes par mois
tendances_ventes = df.groupby('Année-Mois')['Ventes'].sum()
tendances_ventes.index = tendances_ventes.index.to_timestamp()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tendances_ventes.index, tendances_ventes.values, marker='o', color='#1f77b4', linewidth=2)

ax.set_title("Tendances Mensuelles des Ventes (Graphique Interactif Matplotlib)")
ax.set_xlabel("Chronologie (Mois)")
ax.set_ylabel("Volume Global des Ventes ($)")
plt.grid(True, linestyle='--', alpha=0.6)

# Grâce à %matplotlib notebook, vous pouvez utiliser la barre d'outils en bas du graphique
# pour zoomer, vous déplacer (pan) et sauvegarder des zones spécifiques de l'évolution temporelle.
plt.show()

### B. Répartition des ventes par État (Approche Cartographique / Distribution Spatiale)

In [ ]:
ventes_par_etat = df.groupby('État')['Ventes'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(ventes_par_etat.index, ventes_par_etat.values, color='#aec7e8')

ax.set_title("Volume des Ventes par État")
ax.set_ylabel("Ventes cumulées ($)")
plt.xticks(rotation=45)

# Ajout d'une interactivité simple via les outils natifs de zoom/sélection de la fenêtre Matplotlib
plt.tight_layout()
plt.show()

## 3. Visualisation Élégante avec Seaborn
Pour Seaborn, nous rebasculons sur un affichage statique de haute qualité graphique.

### A. Top 10 des produits les plus performants par Ventes

In [ ]:
# Rebasculer en mode d'affichage statique standard pour un rendu Seaborn optimal
%matplotlib inline
sns.set_theme(style="darkgrid")

top_10_produits = df.groupby('Produit')['Ventes'].sum().sort_values(ascending=False).head(10).reset_index()

plt.figure(figsize=(11, 5))
sns.barplot(data=top_10_produits, x='Ventes', y='Produit', palette='viridis')

plt.title('Top 10 des Produits Générateurs de Chiffre d\'Affaires', fontsize=14, weight='bold')
plt.xlabel('Total des Ventes ($)', fontsize=12)
plt.ylabel('Référence Produit', fontsize=12)
plt.show()

### B. Nuage de points : Analyse d'impact des remises (Discount) sur le Profit

In [ ]:
plt.figure(figsize=(10, 6))

# Utilisation de hue pour distinguer l'impact par État et size pour le volume de vente associé
sns.scatterplot(data=df, x='Remise', y='Profit', hue='État', size='Ventes', sizes=(20, 200), palette='deep', alpha=0.8)

plt.axhline(y=0, color='red', linestyle='--', linewidth=1.5, label='Seuil de rentabilité (0 Profit)')
plt.title('Corrélation entre le Taux de Remise et la Rentabilité (Profit)', fontsize=14, weight='bold')
plt.xlabel('Taux de Remise (Discount)', fontsize=12)
plt.ylabel('Marge / Profit net ($)', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Analyse Comparative & Observations

### Enseignements Métiers obtenus :
1. **Tendances Temporelles (Matplotlib) :** Permet de déceler les cycles de vente et les mois creux. L'interactivity offre la liberté d'isoler une période précise.
2. **Impact de la Remise (Seaborn) :** Le nuage de points montre de manière flagrante que dès que la remise dépasse **20%**, la majorité des transactions basculent sous la ligne rouge du seuil de rentabilité et génèrent des **pertes**.

